In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
%pip install -q dagshub mlflow

In [ ]:
import os
import mlflow
import dagshub

os.environ["DAGSHUB_USER_TOKEN"] = "8440d5c76c2d44c640a061571536a753a4804760"
dagshub.init(repo_owner='skoba23', repo_name='ML_Assignment_2', mlflow=True)

In [ ]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity,   on='TransactionID', how='left')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)

# Load Best Model from Registry

In [ ]:
model_name = "IEEE_Fraud_XGBoost"
model_version = 1                   

model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.sklearn.load_model(model_uri)

print(f"Model loaded: {model_name} v{model_version}")

# Preprocessing Test Data

In [ ]:
for col in test.select_dtypes(include="object").columns:
    test[col] = test[col].astype("category").cat.codes

# numeric NaN → median
num_cols = test.select_dtypes(include=np.number).columns.tolist()
test[num_cols] = test[num_cols].fillna(test[num_cols].median())

print("Test data preprocessed:", test.shape)

# Predict

In [ ]:
transaction_ids = test["TransactionID"]
X_test = test.drop(columns=["TransactionID"])

expected_features = model.feature_names_in_
for col in expected_features:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[expected_features]

preds = model.predict_proba(X_test)[:, 1]
print(f"Predictions done! Min: {preds.min():.4f}, Max: {preds.max():.4f}")

# Generating Submission

In [ ]:
submission = pd.DataFrame({
    "TransactionID": transaction_ids,
    "isFraud": preds
})

submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head(10)

In [ ]:
print(f"Total predictions: {len(submission)}")
print(f"Fraud probability range: {submission['isFraud'].min():.4f} - {submission['isFraud'].max():.4f}")
print(f"Average fraud probability: {submission['isFraud'].mean():.4f}")
print("\nSample:")
print(submission.head())